## this notebook is only for developments and tests.  for the  app use app.py

In [2]:
!pip install "fastapi[standard]"

   ---------------------------------------- 0.0/727.1 kB ? eta -:--:--
   ---------------------------------------- 727.1/727.1 kB 3.7 MB/s  0:00:00

   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   ----------------------------------------  0/13 [sentry-sdk]
   -----------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# imports
from fastapi import FastAPI
from pydantic import BaseModel
import chromadb
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic
from dotenv import load_dotenv
import os

ModuleNotFoundError: No module named 'fastapi'

In [ ]:
# load_dotenv()

load_dotenv()
app = FastAPI()

# load chromabd
chroma_client = chromadb.PersistentClient("../vectorstore/food_poverty_db/")
collection_foodprice = chroma_client.get_collection(name="food_prices")
collection_poverty = chroma_client.get_collection(name="poverty_mpi")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

anthropic_client = Anthropic()

class Question(BaseModel):
    question: str

In [ ]:
# building retriever function -that takes a question, searches ChromaDB
# return top 3 relevant documents from each collection and combining them

def retrieve(question, n_results=3):
    question_embedding = embed_model.encode(question).tolist()
    
    food_docs = collection_foodprice.query(
        query_embeddings = [question_embedding],
        n_results = n_results
        )['documents'][0]
    poverty_docs = collection_poverty.query(
        query_embeddings = [question_embedding],
        n_results = n_results
        )['documents'][0]
    
    return food_docs + poverty_docs

In [ ]:
# RAG pipeline — retrieves context, builds prompt, calls Claude, returns answer

conversation_history = []

def ask(question):
    docs = retrieve(question)
    context_text = '\n'.join(docs)
    
    prompt = f"""Context: {context_text}
    Question: {question}
    Anwser based only on context provided."""
    
    conversation_history.append({'role': 'user', 'content': prompt})
    
    response = anthropic_client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens=1024,
        system = """You are a humanitarian data analyst assistant. 
            Answer questions based only on the provided context. 
            Be precise with numbers and facts. 
            Always cite specific figures when available.
            """,
        messages=[{'role': 'user', 'content': prompt}]
    )
    answer = response.content[0].text
    conversation_history.append({'role': 'assistant', 'content': answer})
    return answer


In [ ]:
# POST /ask — main endpoint, takes question, returns AI answer
@app.post("/ask")
def ask_endpoint(body: Question):
    answer = ask(body.question)
    return {"question": body.question, "answer": answer}

In [ ]:
# GET / — health check endpoint
@app.get("/")
def root():
    return {"status": "RAG API is running"}

## FastAPI Endpoint — Humanitarian RAG API

This notebook documents the FastAPI application that serves the conversational 
RAG pipeline as a REST API endpoint.

### What This Does
Wraps the full RAG pipeline (retrieval + LangChain + Claude) into a REST API 
so anyone can query the humanitarian data system with a simple HTTP request.

### How To Run The Server
Open a terminal, activate venv311, navigate to src/ and run:

```bash
source /c/Users/HP/Projects/agentic-ai/practice/venv311/Scripts/activate
cd /c/Users/HP/Projects/agentic-ai/RAG-Humanitarian-risk-analysis-India/src
uvicorn app:app --reload
```

### Endpoints
- `GET /` — Health check, confirms API is running
- `POST /ask` — Takes a question, returns AI-generated answer

### Interactive Docs
Once server is running, visit:
`http://127.0.0.1:8000/docs` — Swagger UI to test endpoints in browser

### Inputs & Outputs
Input: `{"question": "What is the poverty situation in Bihar?"}`  
Output: `{"question": "...", "answer": "..."}`

In [1]:
# test code
import requests

url = "http://127.0.0.1:8000/ask"
payload = {"question": "What is the poverty situation in Bihar?"}

response = requests.post(url, json=payload)
print(response.json())

{'question': 'What is the poverty situation in Bihar?', 'answer': '# Poverty Situation in Bihar\n\nBased on the available data, Bihar has **multiple poverty levels** with varying Multidimensional Poverty Index (MPI) scores:\n\n1. **52.41%** of the population lives in poverty with an MPI score of **0.2475**\n\n2. **77.37%** of the population lives in poverty with an MPI score of **0.4484**\n\n3. **34.66%** of the population lives in poverty with an MPI score of **0.1544**\n\nThese figures suggest that Bihar experiences significant poverty challenges across different measures and severity levels. The varying percentages likely reflect different methodologies or time periods of measurement, with the highest proportion (77.37%) indicating substantial poverty when measured by certain criteria.'}
